<a href="https://colab.research.google.com/github/alexander-toschev/cv-course/blob/main/Tasks/Task6_Semantic.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

# DO NOT MODIFY  !!!
# DO NOT EXECUTE !!!
!pip install --upgrade gspread pandas google-auth
import pandas as pd
import gspread
from google.colab import auth
from google.auth import default
from IPython.display import display
import random
# Authenticate and create the PyDrive client.
auth.authenticate_user()
creds, _ = default()
gc = gspread.authorize(creds)

In [ ]:
# FILL THIS
student_name = "ELON MUSK"
group_id = "11-101"

In [ ]:
# DO NOT MODIFY  !!!
# DO NOT EXECUTE !!!
SPREADSHEET_URL = "https://docs.google.com/spreadsheets/d/1VIp1PdTVfR4rWP44YYWzz58hVf-_wLpX4NwXmcG7MNo/edit?usp=sharing"
sh = gc.open_by_url(SPREADSHEET_URL)
worksheet = sh.sheet1
score = 0
# Ensure header row exists
if not worksheet.get_all_values():
    worksheet.append_row(["Student Name", "Group","TaskID", "Score"])


In [ ]:
# MAIN NOTEBOOK GOES HERE
# DO NOT MODIFY  !!!
# DO NOT EXECUTE !!!
task_id = "Task6_Semantic"
score = 0
max_score = 20

# 🏡 Домашнее задание: Scene Graph Generation + Link Prediction


## 🎯 Цели:
- Построить сценограф изображения.
- Добавить новые признаки пар объектов.
- Предсказать связи между объектами.
- Оценить качество модели.


## 1. Установка библиотек

In [ ]:

!pip install torch torchvision matplotlib networkx scikit-learn -q

import torch
import torchvision
from torchvision.models.detection import fasterrcnn_resnet50_fpn
from torchvision.transforms import functional as F
from PIL import Image
import matplotlib.pyplot as plt
import networkx as nx
import random
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier


## 2. Загрузка модели и данных

In [ ]:

model = fasterrcnn_resnet50_fpn(pretrained=True)
model = model.eval()

# Загрузите или замените изображение по желанию
!wget https://upload.wikimedia.org/wikipedia/commons/9/9a/Pug_600.jpg -O test_image.jpg
image = Image.open('test_image.jpg').convert("RGB")
img_tensor = F.to_tensor(image).unsqueeze(0)


## 3. Детекция объектов

In [ ]:

with torch.no_grad():
    prediction = model(img_tensor)[0]

threshold = 0.8
boxes = prediction['boxes'][prediction['scores'] > threshold]
labels = prediction['labels'][prediction['scores'] > threshold]

COCO_INSTANCE_CATEGORY_NAMES = [
    '__background__', 'person', 'bicycle', 'car', 'motorcycle', 'airplane', 'bus',
    'train', 'truck', 'boat', 'traffic light', 'fire hydrant', 'stop sign', 'parking meter',
    'bench', 'bird', 'cat', 'dog', 'horse', 'sheep', 'cow', 'elephant', 'bear', 'zebra',
    'giraffe', 'backpack', 'umbrella', 'handbag', 'tie', 'suitcase', 'frisbee', 'skis',
    'snowboard', 'sports ball', 'kite', 'baseball bat', 'baseball glove', 'skateboard',
    'surfboard', 'tennis racket', 'bottle', 'wine glass', 'cup', 'fork', 'knife', 'spoon',
    'bowl', 'banana', 'apple', 'sandwich', 'orange', 'broccoli', 'carrot', 'hot dog',
    'pizza', 'donut', 'cake', 'chair', 'couch', 'potted plant', 'bed', 'dining table',
    'toilet', 'tv', 'laptop', 'mouse', 'remote', 'keyboard', 'cell phone', 'microwave',
    'oven', 'toaster', 'sink', 'refrigerator', 'book', 'clock', 'vase', 'scissors',
    'teddy bear', 'hair drier', 'toothbrush'
]

# Вывод объектов
for idx, box in enumerate(boxes):
    print(f"Object {idx}: {COCO_INSTANCE_CATEGORY_NAMES[labels[idx]]} - Box: {box.tolist()}")

n_objects = len(labels)
assert n_objects >= 2, "❌ Нужно минимум 2 объекта!"
score+=5


## 4. Формирование признаков пар объектов (Ваш код)

In [ ]:

# TODO: Добавьте новые признаки: ширина/высота объектов + расстояние между центрами

features = []
pairs = []

for i in range(n_objects):
    for j in range(n_objects):
        if i != j:
            box1, box2 = boxes[i], boxes[j]
            label1, label2 = labels[i].item(), labels[j].item()
            w1 = box1[2] - box1[0]
            h1 = box1[3] - box1[1]
            w2 = box2[2] - box2[0]
            h2 = box2[3] - box2[1]
            center_x1 = (box1[0] + box1[2]) / 2
            center_y1 = (box1[1] + box1[3]) / 2
            center_x2 = (box2[0] + box2[2]) / 2
            center_y2 = (box2[1] + box2[3]) / 2
            delta_x = center_x2 - center_x1
            delta_y = center_y2 - center_y1
            feature = [label1, label2, w1.item(), h1.item(), w2.item(), h2.item(), delta_x.item(), delta_y.item()]
            features.append(feature)
            pairs.append((i, j))

features = np.array(features)

assert features.shape[1] >= 6, "❌ Вектора признаков должны быть минимум 6 признаков!"
score+=5


## 5. Разметка связей вручную

In [ ]:

# TODO: Разметьте вручную, где есть связь (1) или нет (0)

random.seed(42)
relations = [random.choice([0, 1]) for _ in range(len(features))]

X_train, X_test, y_train, y_test = train_test_split(features, relations, test_size=0.3, random_state=42)


## 6. Обучение модели

In [ ]:

clf = MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=500, random_state=42)
clf.fit(X_train, y_train)

print(f"Train Accuracy: {clf.score(X_train, y_train):.2f}")
print(f"Test Accuracy: {clf.score(X_test, y_test):.2f}")


## 7. Построение сцено-графа

In [ ]:

G = nx.DiGraph()

for idx, label in enumerate(labels):
    G.add_node(idx, label=COCO_INSTANCE_CATEGORY_NAMES[label])

predicted_relations = clf.predict(features)

for idx, (i, j) in enumerate(pairs):
    if predicted_relations[idx] == 1:
        G.add_edge(i, j, relation="predicted")

plt.figure(figsize=(12, 9))
pos = nx.spring_layout(G)
labels_node = nx.get_node_attributes(G, 'label')
nx.draw(G, pos, with_labels=True, labels=labels_node, node_color='lightgreen', node_size=2000, font_size=10, arrows=True)
edge_labels = nx.get_edge_attributes(G, 'relation')
nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_labels, font_color='red')
plt.title("Scene Graph with Predicted Relations")
plt.show()


## 8. Проверочные тесты

In [ ]:

# Проверка объектов
assert n_objects >= 2, "❌ Нужно минимум 2 объекта!"
score +=2.5

# Проверка признаков
assert features.shape[1] >= 6, "❌ Признаков должно быть минимум 6!"
score +=2.5
# Проверка графа
assert len(G.nodes) == n_objects, "❌ Не все объекты добавлены в граф!"
score +=2.5
# Проверка связей
assert len(G.edges) > 0, "❌ Нет предсказанных связей!"
score +=2.5
print("✅ Все тесты пройдены! Отличная работа!")


In [ ]:

# DO NOT MODIFY  !!!
# DO NOT EXECUTE !!!
# Save the result to Google Sheets
from datetime import datetime

# Get current date and time
now = datetime.now()
timestamp = now.strftime("%Y-%m-%d %H:%M:%S")
worksheet.append_row([student_name,group_id, task_id, score, timestamp])

print(f"Test completed! {student_name}, your score is {score}/{max_score}.")